# N-Queens Algorithm Comparison

This notebook compares three different algorithmic approaches to solving the N-Queens problem:
1. **Genetic Algorithm** (Greedy Evolution)
2. **Las Vegas Algorithm**
3. **Monte Carlo Algorithm**

## Setup

In [ ]:
# Add parent directory to path
import sys
sys.path.insert(0, '..')

import numpy as np
import time
from src.n_queens import Chromosome, EvolManager, visualization

# Set random seed for reproducibility
np.random.seed(42)

## 1. Genetic Algorithm (Greedy Evolution)

The genetic algorithm evolves a population of chromosomes over multiple generations:
- Selection based on fitness
- Mutation through swap operations
- Elitism to preserve best solutions

### Running the Genetic Algorithm

In [ ]:
# Configure genetic algorithm for 12-Queens
ga_manager = EvolManager(
    genes_per_chrom=12,
    pop=1000,
    generations=50,
    reproductive_pool_size=800,
    offspring=500
)

# Run evolution
print("Running Genetic Algorithm for 12-Queens...\n")
ga_manager.greedy_evolution()

### Visualizing Solutions

In [ ]:
# Display found solutions
ga_solutions = ga_manager.get_solutions()
print(f"\nGenetic Algorithm found {len(ga_solutions)} unique solutions!\n")

# Visualize first 6 solutions
if ga_solutions:
    visualization.render_multiple_solutions(
        ga_solutions,
        max_display=6
    )

### Solution Timeline

In [ ]:
# Plot when solutions were discovered
solution_times = ga_manager.get_solution_times()
if len(solution_times) > 0:
    visualization.plot_solution_distribution(
        solution_times,
        title="Genetic Algorithm: Solution Discovery Timeline (12-Queens)"
    )

## 2. Las Vegas Algorithm

The Las Vegas algorithm uses pure random search:
- Generates random configurations repeatedly
- Guarantees correctness (only returns valid solutions)
- No predetermined runtime

### Running Las Vegas

In [ ]:
# Configure Las Vegas for 12-Queens
lv_manager = EvolManager(
    genes_per_chrom=12,
    pop=100,  # Not used, but required
    generations=100  # Not used, but required
)

print("Running Las Vegas Algorithm (500,000 attempts)...\n")
start_time = time.time()
lv_manager.las_vegas(500000)
lv_time = time.time() - start_time

lv_solutions = lv_manager.get_solutions()
print(f"\nLas Vegas found {len(lv_solutions)} unique solutions in {lv_time:.2f} seconds")

### Las Vegas Solutions

In [ ]:
# Display solutions
lv_manager.show_solutions()

# Visualize first solution
if lv_solutions:
    visualization.render_board(
        lv_solutions[0].get_positions(),
        title="Las Vegas: First Solution Found"
    )

## 3. Monte Carlo Algorithm

The Monte Carlo algorithm estimates the probability of finding solutions:
- Fixed number of random samples
- Returns ratio of solutions to attempts
- Useful for understanding problem difficulty

### Running Monte Carlo

In [ ]:
# Configure Monte Carlo for different board sizes
board_sizes = [8, 10, 12]
mc_results = {}

print("Running Monte Carlo Algorithm...\n")

for n in board_sizes:
    mc_manager = EvolManager(
        genes_per_chrom=n,
        pop=100,
        generations=100
    )
    
    start_time = time.time()
    probability = mc_manager.montecarlo(100000)
    mc_time = time.time() - start_time
    
    mc_results[n] = {
        'probability': probability,
        'time': mc_time
    }
    
    print(f"{n}-Queens: Solution probability = {probability:.6f} ({probability*100:.4f}%)")
    print(f"           Time: {mc_time:.2f} seconds\n")

## Algorithm Comparison

### Performance Summary

In [ ]:
# Collect performance data
import pandas as pd

comparison_data = {
    'Algorithm': ['Genetic Algorithm', 'Las Vegas', 'Monte Carlo'],
    'Solutions Found': [
        len(ga_solutions),
        len(lv_solutions),
        f"{mc_results[12]['probability']*100:.4f}% probability"
    ],
    'Time (seconds)': [
        f"{solution_times[-1] if len(solution_times) > 0 else 0:.2f}",
        f"{lv_time:.2f}",
        f"{mc_results[12]['time']:.2f}"
    ],
    'Approach': [
        'Evolutionary',
        'Random (guaranteed correct)',
        'Random (probabilistic)'
    ]
}

df = pd.DataFrame(comparison_data)
print("\n" + "="*70)
print("ALGORITHM COMPARISON (12-Queens Problem)")
print("="*70 + "\n")
print(df.to_string(index=False))
print("\n" + "="*70)

### Visual Comparison

In [ ]:
# Create performance comparison chart
algorithms = ['Genetic\nAlgorithm', 'Las Vegas', 'Monte Carlo']
times = [
    solution_times[-1] if len(solution_times) > 0 else 0,
    lv_time,
    mc_results[12]['time']
]
solutions = [
    len(ga_solutions),
    len(lv_solutions),
    int(mc_results[12]['probability'] * 100000)  # Approximate count from probability
]

visualization.plot_performance_comparison(
    algorithms,
    times,
    solutions
)

## Conclusions

### Genetic Algorithm
**Strengths:**
- Finds many unique solutions
- Relatively fast for the first solution
- Continues to find diverse solutions over time

**Weaknesses:**
- Requires parameter tuning
- More complex implementation

### Las Vegas Algorithm
**Strengths:**
- Simple implementation
- Guarantees correctness
- No parameters to tune

**Weaknesses:**
- Slower to find solutions
- Unpredictable runtime
- Less efficient for larger N

### Monte Carlo Algorithm
**Strengths:**
- Provides probability estimates
- Useful for understanding problem difficulty
- Predictable runtime

**Weaknesses:**
- Doesn't guarantee finding solutions
- Only provides statistical information
- Less useful when solutions are needed